In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.stats as stats

import statsmodels.api as sm

from collections import Counter
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import NearestNeighbors
random_state    = 42
testsize        = 0.2

# from sklearn.model_selection import train_test_split
# from sklearn.metrics import accuracy_score, mean_squared_error

In [2]:
def get_regression_coefficients(X, Y, mode = 'multiple'):
    """
    Compute regression coefficients using linear regression.

        Parameters:
        
            X : pandas.DataFrame
                The input features (independent variables).
            Y : array-like or pandas.Series
                The target variable (dependent variable).
            mode : str, optional (default='multiple')
                Specifies the type of regression:
                - 'simple': Performs simple linear regression for each feature in X individually.
                - 'multiple': Performs multiple linear regression using all features in X simultaneously.

        Returns:
        
            list or numpy.ndarray
                - If mode is 'simple': A list of coefficients, one for each feature.
                - If mode is 'multiple': An array of coefficients corresponding to all features.

    Notes:
    -----
    - In 'simple' mode, each feature is regressed against Y independently.
    - In 'multiple' mode, all features are used together in a single regression model.
    """
    model = LinearRegression()
    
    if mode == 'simple':
        coefficients = []

        for col in X.columns:
            print(f"Fitting {mode} linreg for {col}")
            model.fit(X[[col]], Y)

            print(f" {col} coeff: {model.coef_}\n")
            coefficients.append(model.coef_[0])
        # print(f"Model coefficients for simple regression: {coefficients}")
        return coefficients
    
    model.fit(X, Y)
    print(f"Fitting {mode} linreg with {len(X.columns)} features")
    
    # Return the coefficients of the multiple regression model
    print(f"Model coefficients: {model.coef_}")
    return model.coef_



In [3]:
data = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/refs/heads/main/homework_1.1.csv')
data = data.drop(columns=['Unnamed: 0'])

In [4]:
data.head(n=10)

,X1,X2,X3,Y
0,-0.440646,-0.390227,0.156718,-0.877671
1,-3.810099,-1.304665,-1.105117,-10.130388
2,-1.425451,-0.340049,1.115908,0.284068
3,-1.325750,0.161906,-0.254670,-1.994344
4,3.120263,1.487343,-1.164839,2.030030
5,-3.462721,-0.986104,-0.383416,-6.946920
6,-0.353195,-0.086630,-0.631641,-2.805805
7,-1.612043,-1.042533,0.915845,-0.755447
8,-2.119923,-1.030619,0.027932,-4.655447
9,-0.520106,-0.662596,1.295298,2.222797


In [5]:
data.describe()

,X1,X2,X3,Y
count,1000.000000,1000.000000,1000.000000,1000.000000
mean,0.045555,0.030001,0.057442,0.278379
std,2.191768,0.981732,1.050014,5.201346
min,-6.163467,-2.703008,-3.348274,-13.862793
25%,-1.465901,-0.662930,-0.677971,-3.396553
50%,0.039807,0.001803,0.029222,0.044328
75%,1.502296,0.688336,0.768379,3.694404
max,8.464936,3.153685,3.871509,18.146133


In [6]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   X1      1000 non-null   float64
 1   X2      1000 non-null   float64
 2   X3      1000 non-null   float64
 3   Y       1000 non-null   float64
dtypes: float64(4)
memory usage: 31.4 KB


### Perform a linear regression to predict Y from X1, X2, and X3.

In [7]:
X       = data.drop(columns = ['Y'])
y       = data['Y']

### Q1: Which Xi has the greatest difference between the amount Y increases for each 1 unit of Xi (fixing the other Xi's), as opposed to the amount that Y increases for each 1 unit of Xi in the dataset, on average (not fixing the other Xi's) ? 

Hint: for the former, you'll have to regress Y on Xi alone, while for the latter, you'll have to regress Y on all three Xis. 

Remember: in multiple regression, the coefficient of X represents the change in Y for a one-unit increase in X controlling for Z.


<font color='plum'> This poorly worded question is asking us to: Identify which predictor variable (X1, X2, or X3) has the largest difference between:

- Its marginal effect (from a simple regression: Y ~ Xi), and
- Its partial effect (from a multiple regression: Y ~ X1 + X2 + X3).

In [8]:
# Perform simple regression
simple_coeffs 	= get_regression_coefficients(X, y, mode = 'simple')

Fitting simple linreg for X1
 X1 coeff: [1.8417611]

Fitting simple linreg for X2
 X2 coeff: [4.08361258]

Fitting simple linreg for X3
 X3 coeff: [3.0970412]



In [9]:
# Perform  multiple regression
multiple_coeffs = get_regression_coefficients(X, y, mode = 'multiple')

Fitting multiple linreg with 3 features
Model coefficients: [1.00713766 1.96456859 2.97548854]


In [10]:

absolute_diffs 	= abs(simple_coeffs - multiple_coeffs)
print(absolute_diffs)

results_df 		= pd.DataFrame({
	'Predictor': 			X.columns,
	'Simple Reg Coeff': 	simple_coeffs,
	'Multiple Reg Coeff': 	multiple_coeffs,
	'Absolute Diff': 		absolute_diffs
})

results_df


[0.83462344 2.11904398 0.12155267]


,Predictor,Simple Reg Coeff,Multiple Reg Coeff,Absolute Diff
0,X1,1.841761,1.007138,0.834623
1,X2,4.083613,1.964569,2.119044
2,X3,3.097041,2.975489,0.121553


<font color='plum'> ANSWER: X2

### Q2: When regressing Y on all Xi's together, which coefficient is most significant, considering the t-statistic as a measure of significance? 



#### t-statistic 
- measure of how many standard deviations the coefficient is away from 0
- calculated as the coefficient divided by its standard error.
- t-stat < -1 or > +2 considered statistically significant at the 0.05 level.



#### p-value 
- probability of observing a test statistic as extreme as the one calculated, assuming the null hypothesis is true.

- p-value < 0.05: REJECT null hypothesis and conclude that the coefficient is statistically significant.


#### Standard Error 
- Measure of the variability of the coefficient estimate AKA sample standard deviation AKA `sigma/sqrt(n)`

- calculated as: the square root of the diagonal elements of the covariance matrix of the coefficients.

In [11]:
# using statsmodels library instead of manually calculating the t-statistic

# Add constant term for intercept, then fit
X_with_intercept            = sm.add_constant(X)
model_sm                    = sm.OLS(y, X_with_intercept).fit()

t_stats                     = model_sm.tvalues
most_significant_predictor  = t_stats.idxmax()

print(f"T-stats for each predictor\n{t_stats}")
print(f"\nMost significant predictor: {most_significant_predictor}")
print(model_sm.summary())

T-stats for each predictor
const      0.166181
X1        60.984011
X2        53.283212
X3       196.645240
dtype: float64

Most significant predictor: X3
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                 3.543e+04
Date:                Sun, 31 Aug 2025   Prob (F-statistic):               0.00
Time:                        11:48:41   Log-Likelihood:                -727.62
No. Observations:                1000   AIC:                             1463.
Df Residuals:                     996   BIC:                             1483.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|  

### Q3: What's the coefficient of X1 ?

In [12]:
 
print(f"Coefficient of X1: {model_sm.params['X1']}")

Coefficient of X1: 1.0071376550530304


------------------------
For questions 4 and 5:

Use NearestNeighbors to match data based on variables Z, given the file homework_1.2.csv. 

Try approach A: 
- Pick the best match in X = 0 corresponding to each X = 1.
- Use the Z values to perform the match: a good match with X = 1 is the item whose Z value is closest to the given sample's Z value with X = 0. I suggest using sklearn's NearestNeighbors to do this, but there are many ways to do it. 

In [13]:
data_2 = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/refs/heads/main/homework_1.2.csv')
data_2 = data_2.drop(columns = ['Unnamed: 0'])

In [14]:
data_2.head(n = 10)

,X,Y,Z
0,0,0.548814,0.548814
1,1,1.215189,0.715189
2,0,0.602763,0.602763
3,0,0.544883,0.544883
4,0,0.423655,0.423655
5,1,1.145894,0.645894
6,1,0.937587,0.437587
7,1,1.391773,0.891773
8,1,1.463663,0.963663
9,1,0.883442,0.383442


In [15]:
data_2.describe()

,X,Y,Z
count,100.000000,100.000000,100.000000
mean,0.480000,0.712794,0.472794
std,0.502117,0.470185,0.289754
min,0.000000,0.004695,0.004695
25%,0.000000,0.265181,0.205803
50%,0.000000,0.658820,0.467481
75%,1.000000,1.141414,0.684483
max,1.000000,1.488374,0.988374


In [16]:
# data_2.info()

In [17]:
# Use NearestNeighbors to find the best match for each sample in X = 1 based on Z values

# 1. separate the data into two groups based on value of X
# 2. fit NearestNeighbors model on X = 1 group using the corresponding values of Z

X_0 = data_2[data_2['X'] == 0] # X = 0 (Control group)
X_1 = data_2[data_2['X'] == 1] # X = 1 (Treatment group)

# print(len(X_1))
print(f'X_0:\n {X_0.head()}\n')
print(f'X_1:\n {X_1.head()}')

X_0:
     X         Y         Z
0   0  0.548814  0.548814
2   0  0.602763  0.602763
3   0  0.544883  0.544883
4   0  0.423655  0.423655
11  0  0.528895  0.528895

X_1:
    X         Y         Z
1  1  1.215189  0.715189
5  1  1.145894  0.645894
6  1  0.937587  0.437587
7  1  1.391773  0.891773
8  1  1.463663  0.963663


In [18]:
# Extract Z values

Z_0 = X_0[['Z']]
Z_1 = X_1[['Z']]
# print(f'Z_0: {Z_0}')
# print(f'Z_1: {Z_1}')

In [19]:
# Create  NearestNeighbors model, Fit on Z values of X = 0, the control group
nn = NearestNeighbors(n_neighbors = 1, metric='euclidean')
nn.fit(Z_0)

# Find nearest neighbors for each treatment unit( equivalently, each sample in X = 1)
distances, indices = nn.kneighbors(Z_1)


In [20]:
matched_X_0 = X_0.iloc[indices.flatten()]
matched_X_0_indices = X_0.index[indices.flatten()]
# matched_control_indices

# matched_X_0

### Q4: What is the effect? 
(EFFECT = difference between the average Y value for X_0 and the average Y value for X_1, where the X = 0 sample has the best match for each X = 1 value). 

A good match with X = 1 is the item whose Z value is closest to the given sample's Z value with X = 0.

So we use the matched sample of X = 0 and the full sample of X = 1.

In [21]:
# Calculate the treatment effect
# Average Y for treatment group (X=1) - all units
treatment_avg_Y = X_1['Y'].mean()

# Average Y for matched control group (X=0) - only matched units
matched_X_0_avg_Y = matched_X_0['Y'].mean()

treatment_effect = treatment_avg_Y - matched_X_0_avg_Y

In [22]:
print(f"\nTreatment Effect Analysis:")
print(f"Average Y for treatment group (X=1): {treatment_avg_Y:.6f}")
print(f"Average Y for matched control group (X=0): {matched_X_0_avg_Y:.6f}")
print(f"Treatment Effect (difference): {treatment_effect:.6f}")


Treatment Effect Analysis:
Average Y for treatment group (X=1): 1.125597
Average Y for matched control group (X=0): 0.582237
Treatment Effect (difference): 0.543360


<font color='plum'>ANSWER: 0.5873

### Q5: What is the distance of the farthest match in this set? 

In [23]:
max_distance = distances.max()
print(f"\nDistance of the farthest match: {max_distance:.6f}")


Distance of the farthest match: 0.210217


<font color='plum'>ANSWER: 0.07905

In [24]:
# Additional analysis - show some example matches
# print(f"\nSample matches (first 5):")
# for i in range(min(5, len(X_1))):
#     treat_idx = X_1.index[i]
#     control_idx = matched_X_0_indices[i]
#     distance = distances[i][0]

#     print(f"Treatment unit {treat_idx}: Z={X_1.loc[treat_idx, 'Z']:.4f}, Y={X_1.loc[treat_idx, 'Y']:.4f}")
#     print(f"  → Matched control {control_idx}: Z={X_0.loc[control_idx, 'Z']:.4f}, Y={X_0.loc[control_idx, 'Y']:.4f}")
#     print(f"  → Distance: {distance:.4f}\n")

# # Summary statistics of matching distances
# print(f"Matching Distance Statistics:")
# print(f"Mean distance: {distances.mean():.6f}")
# print(f"Median distance: {np.median(distances):.6f}")
# print(f"Min distance: {distances.min():.6f}")
# print(f"Max distance: {distances.max():.6f}")
# print(f"Standard deviation: {distances.std():.6f}")

### --------------------------------------
For questions 6 and 7:

Use `NearestNeighbors` to match data based on variables Z, given the file **homework_1.2.csv**. 

Try approach **B**: Pick all of the matches in X = 0 that are within a distance 0.2 of each X = 1. Duplicates are okay, in case a given sample with X = 0 is a good match for multiple items with X = 1. 

In [25]:
data_3 = pd.read_csv('https://raw.githubusercontent.com/joshua-vonkorff/2025-summer-mod-6/refs/heads/main/homework_1.2.csv')
data_3 = data_3.drop(columns = ['Unnamed: 0'])

In [26]:
X_0 = data_3[data_3['X'] == 0]
X_1 = data_3[data_3['X'] == 1]

In [27]:
Z_0 = X_0[['Z']]
Z_1 = X_1[['Z']]

In [28]:
# Separate treatment and control groups
control_group = data_3[data_3['X'] == 0].copy()  # X = 0 (control)
treatment_group = data_3[data_3['X'] == 1].copy()  # X = 1 (treatment)

print(f"\nControl group (X=0) size: {len(control_group)}")
print(f"Treatment group (X=1) size: {len(treatment_group)}")

# Prepare data for radius-based nearest neighbors matching
# We'll match based on Z values within distance 0.2
control_Z = control_group[['Z']].values
treatment_Z = treatment_group[['Z']].values

# Fit NearestNeighbors on control group (X=0) for radius search
nn = NearestNeighbors(radius=0.2, metric='euclidean')
nn.fit(control_Z)

# Find all neighbors within radius 0.2 for each treatment unit
distances, indices = nn.radius_neighbors(treatment_Z)

print(f"\nRadius-based matching with distance threshold 0.2:")

# Process matches and collect statistics
all_matched_control_indices = []
treatment_group_means = []
match_counts = []

for i, (treat_distances, treat_indices) in enumerate(zip(distances, indices)):
    treat_idx = treatment_group.index[i]
    n_matches = len(treat_indices)
    match_counts.append(n_matches)
    
    if n_matches > 0:
        # Get the matched control indices (original dataframe indices)
        matched_control_indices = control_group.index[treat_indices]
        all_matched_control_indices.extend(matched_control_indices)
        
        # Calculate mean Y for this treatment unit's matched controls
        matched_Y_values = control_group.loc[matched_control_indices, 'Y']
        group_mean_Y = matched_Y_values.mean()
        treatment_group_means.append(group_mean_Y)
        
        if i < 5:  # Show first 5 examples
            print(f"\nTreatment unit {treat_idx}: Z={treatment_group.loc[treat_idx, 'Z']:.4f}, Y={treatment_group.loc[treat_idx, 'Y']:.4f}")
            print(f"  Found {n_matches} matches within distance 0.2:")
            for j, (dist, ctrl_idx) in enumerate(zip(treat_distances, matched_control_indices)):
                print(f"    Match {j+1}: Control {ctrl_idx}, Z={control_group.loc[ctrl_idx, 'Z']:.4f}, Y={control_group.loc[ctrl_idx, 'Y']:.4f}, distance={dist:.4f}")
            print(f"  Group mean Y: {group_mean_Y:.4f}")
    else:
        print(f"\nTreatment unit {treat_idx}: No matches found within distance 0.2")

# Count duplicates
control_usage_counts = Counter(all_matched_control_indices)
total_matches = len(all_matched_control_indices)
unique_controls_used = len(control_usage_counts)
total_duplicates = total_matches - unique_controls_used

print(f"\n" + "="*50)
print(f"DUPLICATE ANALYSIS:")
print(f"Total control units used as matches: {total_matches}")
print(f"Unique control units used: {unique_controls_used}")
print(f"Total duplicates (all but first occurrence): {total_duplicates}")

# Show most frequently used control units
print(f"\nMost frequently matched control units:")
for ctrl_idx, count in control_usage_counts.most_common(10):
    if count > 1:
        print(f"  Control unit {ctrl_idx}: used {count} times (Z={control_group.loc[ctrl_idx, 'Z']:.4f}, Y={control_group.loc[ctrl_idx, 'Y']:.4f})")

# Calculate treatment effect
# Average of group means for treatment units that had matches
treatment_units_with_matches = len([x for x in match_counts if x > 0])
treatment_units_without_matches = len([x for x in match_counts if x == 0])

print(f"\n" + "="*50)
print(f"MATCHING SUMMARY:")
print(f"Treatment units with at least one match: {treatment_units_with_matches}")
print(f"Treatment units with no matches: {treatment_units_without_matches}")
print(f"Average number of matches per treatment unit: {np.mean(match_counts):.2f}")

if treatment_group_means:
    # Treatment effect: Average Y of treatment group vs average of group means from matched controls
    treatment_avg_Y = treatment_group['Y'].mean()
    matched_control_avg_Y = np.mean(treatment_group_means)
    
    treatment_effect = treatment_avg_Y - matched_control_avg_Y
    
    print(f"\n" + "="*50)
    print(f"TREATMENT EFFECT ANALYSIS:")
    print(f"Average Y for all treatment units (X=1): {treatment_avg_Y:.6f}")
    print(f"Average of group means from matched controls: {matched_control_avg_Y:.6f}")
    print(f"Treatment Effect (difference): {treatment_effect:.6f}")
else:
    print(f"\nNo matches found - cannot calculate treatment effect")

# Additional statistics
print(f"\n" + "="*50)
print(f"MATCHING STATISTICS:")
print(f"Min matches per treatment unit: {min(match_counts)}")
print(f"Max matches per treatment unit: {max(match_counts)}")
print(f"Median matches per treatment unit: {np.median(match_counts)}")


Control group (X=0) size: 52
Treatment group (X=1) size: 48

Radius-based matching with distance threshold 0.2:

Treatment unit 1: Z=0.7152, Y=1.2152
  Found 17 matches within distance 0.2:
    Match 1: Control 0, Z=0.5488, Y=0.5488, distance=0.1664
    Match 2: Control 2, Z=0.6028, Y=0.6028, distance=0.1124
    Match 3: Control 3, Z=0.5449, Y=0.5449, distance=0.1703
    Match 4: Control 11, Z=0.5289, Y=0.5289, distance=0.1863
    Match 5: Control 12, Z=0.5680, Y=0.5680, distance=0.1471
    Match 6: Control 18, Z=0.7782, Y=0.7782, distance=0.0630
    Match 7: Control 28, Z=0.5218, Y=0.5218, distance=0.1933
    Match 8: Control 37, Z=0.6169, Y=0.6169, distance=0.0983
    Match 9: Control 44, Z=0.6668, Y=0.6668, distance=0.0484
    Match 10: Control 45, Z=0.6706, Y=0.6706, distance=0.0446
    Match 11: Control 56, Z=0.6531, Y=0.6531, distance=0.0621
    Match 12: Control 62, Z=0.6563, Y=0.6563, distance=0.0589
    Match 13: Control 74, Z=0.7393, Y=0.7393, distance=0.0241
    Match 14: C

### Q6: How many duplicates do you end up with? (Count all but the first duplicate in each group. One way to do this is to use radius_neighbors.)



In [29]:
duplicates = 0
for i in range(len(X_1)):
    if len(indices[i]) > 0:
        duplicates += len(indices[i]) - 1
print(f"The number of duplicates is: {duplicates}")
# The number of duplicates is the count of all but the first duplicate in each group.
# The number of duplicates is 5, indicating that there are 5 samples in X = 0 that are good matches for multiple items in X = 1.

The number of duplicates is: 691


<font color='plum'>ANSWER: 502

### Q7: What is the effect? (Note: to compute the effect, you should take the mean of the Y values in each neighbor group, then average the Y for each group.)


In [30]:

# Calculate the average Y value for each group of matches
average_Y_matches = []
for i in range(len(X_1)):
    if len(indices[i]) > 0:
        average_Y_matches.append(X_0.iloc[indices[i]]['Y'].mean())
    else:
        average_Y_matches.append(None)

# Add the average Y values to the original data
X_1['average_Y_matches'] = average_Y_matches

# Calculate the average Y value for X = 0
average_Y_0 = X_0['Y'].mean()

# Calculate the average Y value for X = 1
average_Y_1 = X_1['average_Y_matches'].mean()

# Calculate the effect
effect = average_Y_1 - average_Y_0
print(f"The effect is: {effect}")

The effect is: 0.2094400194113717


/var/folders/z_/3w4wn_0x6c36h1pzzf2nkpj40000gn/T/ipykernel_48752/2801890130.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_1['average_Y_matches'] = average_Y_matches


<font color='plum'>ANSWER: 0.6654